# Create 20-Seed Datasets (MedCLIP Embeddings)

Reads **pre-computed coreset IDs** (txt files) for 20 seeds × 3 strategies,
loads the corresponding MedCLIP embeddings from batch NPZ files,
merges with metadata, and saves each dataset as **pickle (.pkl)** first
(memory-safe), then batch-converts to **parquet** at the end.

No sampling-strategy code is needed — the coreset IDs already encode the selection.

In [ ]:
import os
import glob
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm

CONFIG = {
    'NUM_SEEDS': 20,
    'STRATEGIES': {
        5:  'PathologyStratifiedClean',
        9:  'GradMatch',
        11: 'Uncertainty',
    },

    # --- Colab / Google Drive paths ---
    'EMBEDDING_PATH': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/ve_cxr/elixir/batch_*.npz',
    'METADATA_PATH': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv',
    'METADATA_EXTENDED_PATH': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_updatedlabel_wide_processed.csv',
    'PATH_TO_ADMISSIONS': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/admissions.csv',
    'CORESET_IDS_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/coreset-ids',
    'OUTPUT_BASE_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/medclip-20-seeds',

    # Processing settings (same as original notebook)
    'PATIENT_ID_COLUMN': 'subject_id',
    'INSURANCE_COLUMN': 'new_insurance_type',
    'DICOM_PATH_COLUMN': 'dicom_path',
    'STRIP_FILE_EXTENSION': True,
    'BASE_RANDOM_STATE': 42,
    'RESUME_FROM_EXISTING': True,
}

print('Config loaded.')
print(f"Seeds: {CONFIG['NUM_SEEDS']}")
print(f"Strategies: {CONFIG['STRATEGIES']}")
print(f"Output: {CONFIG['OUTPUT_BASE_DIR']}")

In [ ]:
# ============================================================================
# Helper functions
# ============================================================================

def map_insurance_type(insurance):
    """Map insurance values to binary categories (matching reference notebook)."""
    if pd.isna(insurance):
        return None
    insurance_str = str(insurance).strip()
    if insurance_str.lower() == 'private':
        return 'Private'
    elif insurance_str.lower() in ['medicaid', 'medicare']:
        return 'Medicaid_Medicare'
    else:
        return None


def load_embeddings_for_filenames(filenames_set):
    """Load only the embeddings needed for the given filenames from batch NPZ files."""
    emb_dict = {}  # filename -> embedding
    batch_files = sorted(glob.glob(CONFIG['EMBEDDING_PATH']))

    for path in tqdm(batch_files, desc='Loading embeddings'):
        data = np.load(path, allow_pickle=True)
        fnames = data['filenames']
        embeddings = data['embeddings']
        for emb, fname in zip(embeddings, fnames):
            fname_clean = str(fname)
            if fname_clean in filenames_set:
                emb_dict[fname_clean] = emb
                if len(emb_dict) >= len(filenames_set):
                    break
        del data, fnames, embeddings
        gc.collect()
        if len(emb_dict) >= len(filenames_set):
            break

    df = pd.DataFrame([
        {'filename': fn, 'embedding': emb}
        for fn, emb in emb_dict.items()
    ])
    del emb_dict
    gc.collect()
    return df


def load_coreset_ids(seed_idx, strategy_id):
    """Read pre-computed coreset IDs from txt file. Returns set of filenames."""
    strategy_name = CONFIG['STRATEGIES'][strategy_id]
    txt_path = os.path.join(
        CONFIG['CORESET_IDS_DIR'],
        f'seed_{seed_idx}',
        f'coreset-has_pathology-{strategy_id}-{strategy_name}-seed_{seed_idx}.txt'
    )
    if not os.path.exists(txt_path):
        raise FileNotFoundError(f'Coreset IDs file not found: {txt_path}')

    with open(txt_path, 'r') as f:
        ids = {line.strip() for line in f if line.strip()}
    print(f'  Loaded {len(ids)} coreset IDs from {os.path.basename(txt_path)}')
    return ids


print('Helper functions defined.')

In [ ]:
# ============================================================================
# Load & prepare metadata (run once)
# ============================================================================
print('=' * 70)
print('LOADING AND PREPARING METADATA')
print('=' * 70)

# 1. Load main metadata
meta = pd.read_csv(CONFIG['METADATA_PATH'])
print(f'Loaded metadata: {len(meta):,} rows')

# Create filename column from dicom_path
meta['filename'] = meta[CONFIG['DICOM_PATH_COLUMN']].apply(
    lambda x: os.path.basename(x) if pd.notna(x) else None
)
if CONFIG['STRIP_FILE_EXTENSION']:
    meta['filename'] = meta['filename'].apply(
        lambda x: os.path.splitext(x)[0] if pd.notna(x) else None
    )
meta = meta[meta['filename'].notna()].reset_index(drop=True)
print(f'After cleaning filenames: {len(meta):,} rows')

# 2. Load admissions and map insurance
admissions = pd.read_csv(CONFIG['PATH_TO_ADMISSIONS'])
print(f'Loaded admissions: {len(admissions):,} rows')

admissions['new_insurance_type'] = admissions['insurance'].apply(map_insurance_type)
admissions_clean = admissions[admissions['new_insurance_type'].notna()][
    ['subject_id', 'new_insurance_type']
].copy()
admissions_clean = admissions_clean.drop_duplicates(subset=['subject_id'], keep='first')
print(f'Unique patients with valid insurance: {len(admissions_clean):,}')

ins_counts = admissions_clean['new_insurance_type'].value_counts()
for ins_type, count in ins_counts.items():
    print(f'  {ins_type}: {count:,} ({count / len(admissions_clean) * 100:.1f}%)')

# 3. Merge metadata with insurance
meta_before = len(meta)
meta = pd.merge(meta, admissions_clean, on='subject_id', how='inner')
print(f'\nAfter merging with insurance: {len(meta):,} rows (lost {meta_before - len(meta):,})')

# 4. Remove duplicate filenames
meta = meta.drop_duplicates(subset=['filename'], keep='first')
print(f'After removing duplicate filenames: {len(meta):,} rows')

# 5. One image per patient (same random state as original)
meta_prepared = meta.groupby(CONFIG['PATIENT_ID_COLUMN']).sample(
    n=1, random_state=CONFIG['BASE_RANDOM_STATE']
).reset_index(drop=True)
print(f'After one image per patient: {len(meta_prepared):,} rows')

# Index by filename for fast lookup
meta_prepared = meta_prepared.set_index('filename')

print(f'\nMetadata prepared. Unique patients: {meta_prepared[CONFIG["PATIENT_ID_COLUMN"]].nunique():,}')
print('Insurance distribution:')
for ins_type, count in meta_prepared[CONFIG['INSURANCE_COLUMN']].value_counts().items():
    print(f'  {ins_type}: {count:,} ({count / len(meta_prepared) * 100:.1f}%)')

In [ ]:
# ============================================================================
# Main processing loop (loads embeddings per task, memory-safe for Colab)
# Output structure: OUTPUT_BASE_DIR/seed_X/data_type{id}_n{count}.pkl
# ============================================================================
os.makedirs(CONFIG['OUTPUT_BASE_DIR'], exist_ok=True)

results_log = []
total_tasks = CONFIG['NUM_SEEDS'] * len(CONFIG['STRATEGIES'])
task_num = 0

for seed_idx in range(CONFIG['NUM_SEEDS']):
    seed_dir = os.path.join(CONFIG['OUTPUT_BASE_DIR'], f'seed_{seed_idx}')
    os.makedirs(seed_dir, exist_ok=True)

    for strategy_id, strategy_name in CONFIG['STRATEGIES'].items():
        task_num += 1

        # --- Resume support: skip if output pkl already exists ---
        existing = glob.glob(os.path.join(
            seed_dir,
            f'data_type{strategy_id}_n*.pkl'
        ))
        if CONFIG['RESUME_FROM_EXISTING'] and existing:
            print(f'[{task_num}/{total_tasks}] SKIP seed_{seed_idx} / type{strategy_id} '
                  f'(exists: {os.path.basename(existing[0])})')
            continue

        print(f'\n[{task_num}/{total_tasks}] Processing seed_{seed_idx} / '
              f'{strategy_name} (type{strategy_id})')

        # 1. Load coreset IDs
        coreset_ids = load_coreset_ids(seed_idx, strategy_id)

        # 2. Filter metadata to matching filenames
        matching = meta_prepared.index.isin(coreset_ids)
        meta_filtered = meta_prepared[matching].copy()
        meta_filtered = meta_filtered.reset_index()  # bring filename back as column
        print(f'  Metadata matches: {len(meta_filtered):,} / {len(coreset_ids):,} coreset IDs')

        if len(meta_filtered) == 0:
            print('  WARNING: No metadata matches! Skipping.')
            continue

        # 3. Load embeddings sequentially (memory-safe)
        filenames_set = set(meta_filtered['filename'].values)
        emb_df = load_embeddings_for_filenames(filenames_set)
        print(f'  Embeddings loaded: {len(emb_df):,} / {len(filenames_set):,}')

        if len(emb_df) == 0:
            print('  WARNING: No embeddings found! Skipping.')
            del meta_filtered
            gc.collect()
            continue

        # 4. Merge metadata + embeddings, then free inputs immediately
        df_final = pd.merge(meta_filtered, emb_df, on='filename', how='inner')
        del emb_df, meta_filtered
        gc.collect()
        print(f'  Final merged rows: {len(df_final):,}')

        # 5. Save as pickle into seed_X/ subfolder (no array conversion needed)
        n_samples = len(df_final)
        output_filename = f'data_type{strategy_id}_n{n_samples}.pkl'
        output_path = os.path.join(seed_dir, output_filename)
        df_final.to_pickle(output_path)

        file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f'  Saved: seed_{seed_idx}/{output_filename} ({file_size_mb:.1f} MB)')

        # 6. Print insurance distribution
        ins_dist = df_final[CONFIG['INSURANCE_COLUMN']].value_counts()
        dist_str = ' | '.join(f'{k}={v}' for k, v in ins_dist.items())
        print(f'  Insurance: {dist_str}')

        results_log.append({
            'seed': seed_idx,
            'strategy_id': strategy_id,
            'strategy_name': strategy_name,
            'n_samples': n_samples,
            'file_size_mb': round(file_size_mb, 2),
            **{f'ins_{k}': v for k, v in ins_dist.items()},
        })

        del df_final
        gc.collect()

print(f'\n{"=" * 70}')
print(f'DONE. Processed {len(results_log)} new datasets.')

In [ ]:
# ============================================================================
# Summary
# ============================================================================
print('=' * 70)
print('SUMMARY OF ALL GENERATED DATASETS')
print('=' * 70)

all_files = sorted(glob.glob(os.path.join(CONFIG['OUTPUT_BASE_DIR'], 'seed_*', '*.pkl')))
print(f'\nTotal pkl files: {len(all_files)}')
print(f"Expected: {CONFIG['NUM_SEEDS'] * len(CONFIG['STRATEGIES'])} "
      f"(20 seeds x {len(CONFIG['STRATEGIES'])} strategies)\n")

if all_files:
    total_size = sum(os.path.getsize(f) for f in all_files) / (1024 * 1024)
    print(f'Total size: {total_size:.1f} MB\n')
    for f in all_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        rel = os.path.join(os.path.basename(os.path.dirname(f)), os.path.basename(f))
        print(f'  {rel:60s} {size_mb:6.1f} MB')

# Save results log
if results_log:
    log_df = pd.DataFrame(results_log)
    log_path = os.path.join(CONFIG['OUTPUT_BASE_DIR'], 'generation_log.csv')
    log_df.to_csv(log_path, index=False)
    print(f'\nGeneration log saved to: {log_path}')

    print('\nPer-strategy summary:')
    for sid, sname in CONFIG['STRATEGIES'].items():
        strat_rows = log_df[log_df['strategy_id'] == sid]
        if len(strat_rows) > 0:
            avg_n = strat_rows['n_samples'].mean()
            print(f'  Type {sid} ({sname}): {len(strat_rows)} files, avg {avg_n:.0f} samples')

In [ ]:
# ============================================================================
# Convert all pkl files to parquet (one at a time to stay memory-safe)
# Array columns (e.g. embeddings) are stored as list-of-floats for parquet.
# Parquet files go to a SEPARATE directory mirroring the seed_X/ structure.
# ============================================================================
from pathlib import Path

PARQUET_OUTPUT_DIR = CONFIG['OUTPUT_BASE_DIR'].rstrip('/') + '-parquet'


def _is_array_like(x):
    """True if value looks like a numeric array (ndarray, list of numbers), not str/bytes."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return False
    return hasattr(x, "__len__") and hasattr(x, "__getitem__") and not isinstance(x, (str, bytes))


def _to_list(x):
    """Convert array-like to list for parquet (e.g. numpy ndarray -> list of floats)."""
    if hasattr(x, "tolist"):
        return x.tolist()
    return list(x) if _is_array_like(x) else x


def convert_pkl_to_parquet(pkl_path, parquet_path):
    """Load a single pkl, convert array columns to lists, write parquet, free memory."""
    df = pd.read_pickle(pkl_path)
    for col in df.columns:
        try:
            sample = df[col].iloc[0]
            if _is_array_like(sample):
                df[col] = df[col].apply(_to_list)
        except Exception:
            pass
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(parquet_path, index=False)
    size_mb = parquet_path.stat().st_size / (1024 * 1024)
    del df
    gc.collect()
    return size_mb


print('=' * 70)
print('CONVERTING PKL -> PARQUET (one file at a time)')
print(f'Parquet output dir: {PARQUET_OUTPUT_DIR}')
print('=' * 70)

pkl_base = Path(CONFIG['OUTPUT_BASE_DIR'])
pkl_files = sorted(pkl_base.glob('seed_*/*.pkl'))
print(f'Found {len(pkl_files)} pkl files to convert.\n')

converted = 0
for pkl_path in pkl_files:
    # Mirror the seed_X/ structure in the separate parquet dir
    rel = pkl_path.relative_to(pkl_base)
    parquet_path = Path(PARQUET_OUTPUT_DIR) / rel.with_suffix('.parquet')

    if parquet_path.exists():
        print(f'  SKIP (parquet exists): {rel.parent}/{parquet_path.name}')
        continue
    size_mb = convert_pkl_to_parquet(pkl_path, parquet_path)
    print(f'  {rel} -> {parquet_path.name} ({size_mb:.1f} MB)')
    converted += 1

print(f'\nDone. Converted {converted} files. '
      f'Skipped {len(pkl_files) - converted} (already had parquet).')

# Final count
parquet_files = sorted(Path(PARQUET_OUTPUT_DIR).glob('seed_*/*.parquet'))
print(f'Total parquet files: {len(parquet_files)}')